# 🥊 Bangla TTS Shootout — promito Bangla er jonno best model khuji

Compare kori (eki Bangladeshi promito reference + eki text-e):
1. **base IndicF5** (ai4bharat/IndicF5) — strong Bengali phonology, kintu Indian-leaning
2. **ehzawad/indicf5-bangla-tts** — IndicF5, **Bangladeshi Bengali** e fine-tuned

Duitai tomar **BD promito reference** clone kore, tai output BD accent-er dike jhukbe। Shune je-ta best **Bangladeshi promito**, oita app-e boshabo।

> Chatterbox (ekhonkar app) er output tomar kache already ache — tar sathe compare koro.

**Age koro:** Runtime → Change runtime type → **T4 GPU** → Save.

## 1. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), '⚠️ GPU nai! Runtime > Change runtime type > T4 GPU koro.'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Install IndicF5 (~5 min)
IndicF5 repo + faster-whisper (reference transcript er jonno)।

> Note: pip-e alada `f5_tts` thakle IndicF5 er vendored version-ke shadow kore — tai seta uninstall kori.

In [ ]:
!pip -q uninstall -y f5_tts f5-tts 2>/dev/null
!pip -q install git+https://github.com/ai4bharat/IndicF5.git
!pip -q install faster-whisper soundfile
print('Installed ✓  (kono dependency ERROR thakle amake dekhao)')

## 3. Bangladeshi promito reference upload
Ekta **clean 5-15 sec** clip upload koro — **Bangladeshi promito Bangla** (news-anchor / clear speaker, noise na)। Ei accent-i output-e ashbe.

In [ ]:
from google.colab import files
import os
os.makedirs('/content/out', exist_ok=True)
print('Bangladeshi promito reference clip (wav/mp3) upload koro...')
up = files.upload()
src = list(up.keys())[0]
REF = '/content/ref.wav'
# normalize -> mono 24k wav
!ffmpeg -y -i "{src}" -ac 1 -ar 24000 -t 15 {REF} -loglevel error
print('Reference set:', REF)
from IPython.display import Audio; Audio(REF)

## 4. Reference-er transcript (IndicF5 er dorkar)
IndicF5 e reference audio-r **exact transcript** lage। Ami auto-transcribe kore dicchi (faster-whisper, bn)। 
Nicher `REF_TEXT` output-ta **check koro — vul thakle hate thik kore nao** (accuracy quality bharay)।

In [ ]:
from faster_whisper import WhisperModel
wm = WhisperModel('medium', device='cuda', compute_type='int8_float16')
segs, _ = wm.transcribe(REF, language='bn', beam_size=5)
REF_TEXT = ''.join(s.text for s in segs).strip()
del wm; import gc, torch; gc.collect(); torch.cuda.empty_cache()
print('REF_TEXT =', REF_TEXT)
print('\n^ Vul thakle nicher cell e REF_TEXT hate thik kore dao।')

In [ ]:
# Dorkar hole REF_TEXT hate thik koro (auto-transcript vul thakle uncomment kore edit koro):
# REF_TEXT = "tomar reference clip e thik ja bola hoyeche seta ekhane likho"
print('Using REF_TEXT:', REF_TEXT)

## 5. Target promito Bangla text (ja bolabe)
Ekta bhalo promito test-line rakho (conjunct/number shoho — pronunciation test korte)।

In [ ]:
TARGET = "সুপ্রভাত। আজকের আবহাওয়া সম্পর্কে কিছু গুরুত্বপূর্ণ তথ্য জানানো হচ্ছে। তাপমাত্রা ছত্রিশ ডিগ্রি সেলসিয়াস পর্যন্ত উঠতে পারে।"
print(TARGET)

## 6. 🅰️ base IndicF5 (ai4bharat) — with tomar BD reference

In [ ]:
import numpy as np, soundfile as sf
from transformers import AutoModel

def to_f32(a):
    a = np.asarray(a)
    if a.dtype == np.int16:
        a = a.astype(np.float32) / 32768.0
    return a.astype(np.float32).squeeze()

m_base = AutoModel.from_pretrained('ai4bharat/IndicF5', trust_remote_code=True).to('cuda')
wav = m_base(TARGET, ref_audio_path=REF, ref_text=REF_TEXT)
sf.write('/content/out/A_indicf5_base.wav', to_f32(wav), 24000)
print('Saved: A_indicf5_base.wav')
from IPython.display import Audio; Audio('/content/out/A_indicf5_base.wav')

## 7. 🅱️ ehzawad/indicf5-bangla-tts (Bangladeshi fine-tune) — with tomar BD reference

In [ ]:
# VRAM khali kori age
del m_base; import gc, torch; gc.collect(); torch.cuda.empty_cache()

m_bd = AutoModel.from_pretrained('ehzawad/indicf5-bangla-tts', trust_remote_code=True).to('cuda')
wav = m_bd(TARGET, ref_audio_path=REF, ref_text=REF_TEXT)
sf.write('/content/out/B_indicf5_bangladeshi.wav', to_f32(wav), 24000)
print('Saved: B_indicf5_bangladeshi.wav')
from IPython.display import Audio; Audio('/content/out/B_indicf5_bangladeshi.wav')

## 8. Compare + download
Uporer **A** ar **B** shono, ar app-er **Chatterbox** output-er sathe compare koro:
- Kon-ta **Bangladeshi promito** best (conjunct, number, natural intonation)?
- Amake bolo kon-ta jeto — oita app-er Bangla engine hisebe boshiye debo।

In [ ]:
from google.colab import files
for f in ['A_indicf5_base.wav', 'B_indicf5_bangladeshi.wav']:
    files.download('/content/out/' + f)